In [ ]:
# ============================================================
# CELL 1 - UPLOAD AND EXTRACT DATASET
# ============================================================

import os
import zipfile
from google.colab import files

print("Upload your Kaggle dataset ZIP:")

uploaded = files.upload()

zip_file = list(uploaded.keys())[0]

extract_path = "/content/fire_dataset"

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("\nDataset extracted successfully!")
print("Dataset location:", extract_path)

Upload your Kaggle dataset ZIP:


Saving archive (9).zip to archive (9).zip

Dataset extracted successfully!
Dataset location: /content/fire_dataset


In [ ]:
# ============================================================
# CELL 2 - TRAIN MOBILENETV2 MODEL
# ============================================================

import os
import shutil
import random
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2


# ============================================================
# FIND TRAIN AND TEST FOLDERS
# ============================================================

def find_folder(folder_name, search_path):

    for root, dirs, files_ in os.walk(search_path):

        for d in dirs:

            if d.lower() == folder_name.lower():
                return os.path.join(root, d)

    return None


extract_path = "/content/fire_dataset"

train_dir = find_folder("train", extract_path)
test_dir = find_folder("test", extract_path)

print("Train folder:", train_dir)
print("Test folder:", test_dir)


# ============================================================
# CHECK CLASSES
# ============================================================

class_names = sorted([
    d for d in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, d))
])

print("\nClasses:", class_names)

for class_name in class_names:

    folder = os.path.join(train_dir, class_name)

    images = [
        f for f in os.listdir(folder)
        if f.lower().endswith(
            (".jpg", ".jpeg", ".png")
        )
    ]

    print(class_name, ":", len(images), "images")


# ============================================================
# CREATE VALIDATION DATASET
# ============================================================

validation_dir = "/content/validation"

# Remove old validation folder if it exists
if os.path.exists(validation_dir):
    shutil.rmtree(validation_dir)

os.makedirs(validation_dir, exist_ok=True)

random.seed(42)

print("\nCreating validation dataset...")

for class_name in class_names:

    source_folder = os.path.join(
        train_dir,
        class_name
    )

    validation_class_folder = os.path.join(
        validation_dir,
        class_name
    )

    os.makedirs(
        validation_class_folder,
        exist_ok=True
    )

    images = [
        f for f in os.listdir(source_folder)
        if f.lower().endswith(
            (".jpg", ".jpeg", ".png")
        )
    ]

    random.shuffle(images)

    validation_count = int(len(images) * 0.20)

    validation_images = images[:validation_count]

    for image in validation_images:

        source = os.path.join(
            source_folder,
            image
        )

        destination = os.path.join(
            validation_class_folder,
            image
        )

        shutil.move(
            source,
            destination
        )

    print(
        class_name,
        "->",
        validation_count,
        "validation images"
    )

print("\nValidation dataset created!")


# ============================================================
# LOAD DATASETS
# ============================================================

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
    class_names=class_names
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    class_names=class_names
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    class_names=class_names
)


# ============================================================
# PREFETCH
# ============================================================

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)



# ============================================================
# MOBILE NET V2
# ============================================================

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False


# ============================================================
# BUILD MODEL
# ============================================================

model = models.Sequential([

    layers.Input(
        shape=(224, 224, 3)
    ),



    layers.Rescaling(
        1./127.5,
        offset=-1
    ),

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(0.3),

    layers.Dense(
        len(class_names),
        activation="softmax"
    )

])


# ============================================================
# COMPILE
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("\nModel created successfully!")


# ============================================================
# TRAIN
# ============================================================

print("\nStarting training...")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)


# ============================================================
# TEST ACCURACY
# ============================================================

test_loss, test_accuracy = model.evaluate(
    test_ds,
    verbose=1
)

print("\n================================")
print(
    "TEST ACCURACY:",
    round(test_accuracy * 100, 2),
    "%"
)

print(
    "TEST LOSS:",
    round(test_loss, 4)
)

print("================================")


# ============================================================
# SAVE MODEL
# ============================================================

model_path = "/content/fire_smoke_mobilenetv2.keras"

model.save(model_path)

# Save class names too
class_names_path = "/content/class_names.npy"
np.save(class_names_path, np.array(class_names))

print("\nModel saved at:", model_path)
print("Class names saved at:", class_names_path)

Train folder: /content/fire_dataset/Forect Fire/Forest Fire_Dataset/train
Test folder: /content/fire_dataset/Forect Fire/Forest Fire_Dataset/test

Classes: ['fire', 'nofire', 'smoke', 'smokefire']
fire : 410 images
nofire : 410 images
smoke : 410 images
smokefire : 410 images

Creating validation dataset...
fire -> 82 validation images
nofire -> 82 validation images
smoke -> 82 validation images
smokefire -> 82 validation images

Validation dataset created!
Found 1312 files belonging to 4 classes.
Found 328 files belonging to 4 classes.
Found 800 files belonging to 4 classes.

Model created successfully!

Starting training...
Epoch 1/15
41/41 ━━━━━━━━━━━━━━━━━━━━ 88s 2s/step - accuracy: 0.3430 - loss: 1.5162 - val_accuracy: 0.5884 - val_loss: 1.0280
Epoch 2/15
41/41 ━━━━━━━━━━━━━━━━━━━━ 88s 2s/step - accuracy: 0.5877 - loss: 0.9607 - val_accuracy: 0.7500 - val_loss: 0.7559
Epoch 3/15
41/41 ━━━━━━━━━━━━━━━━━━━━ 80s 2s/step - accuracy: 0.7066 - loss: 0.7432 - val_accuracy: 0.7927 - val_l

In [ ]:
# ============================================================
# CELL 3 - PREDICT IMAGE
# ============================================================

import tensorflow as tf
import numpy as np
from google.colab import files

# Load trained model
model = tf.keras.models.load_model(
    "/content/fire_smoke_mobilenetv2.keras"
)

# Load class names
class_names = np.load(
    "/content/class_names.npy",
    allow_pickle=True
).tolist()

IMG_SIZE = (224, 224)


# ============================================================
# UPLOAD IMAGE
# ============================================================

print("Upload an image:")

uploaded_image = files.upload()

image_path = list(uploaded_image.keys())[0]


# ============================================================
# PREPROCESS IMAGE
# ============================================================

img = tf.keras.utils.load_img(
    image_path,
    target_size=IMG_SIZE
)

img_array = tf.keras.utils.img_to_array(img)

img_array = tf.expand_dims(
    img_array,
    0
)


# ============================================================
# PREDICTION
# ============================================================

prediction = model.predict(
    img_array,
    verbose=0
)

predicted_index = np.argmax(
    prediction[0]
)

predicted_class = class_names[
    predicted_index
]

confidence = (
    prediction[0][predicted_index] * 100
)


# ============================================================
# FINAL RESULT
# ============================================================

print("\n================================")
print("       FINAL PREDICTION")
print("================================")

print(
    "Class      :",
    predicted_class.upper()
)

print(
    "Confidence :",
    round(confidence, 2),
    "%"
)

print("================================")

Upload an image:


Saving fire_test_1011.jpg to fire_test_1011.jpg

       FINAL PREDICTION
Class      : FIRE
Confidence : 80.9 %
